# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [2]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [8]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)

La siguiente celda permite jugar al juego no determinista.

In [ ]:
env = gym.make('FrozenLake-v1', map_name="4x4", render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)

Controles:
W - Arriba
S - Abajo
A - Izquierda
D - Derecha
Q - Salir
Presione cualquier tecla para empezar...
Acción: 1, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: True
¡Episodio terminado! Recompensa final: 0
Acción: 2, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: True
¡Episodio terminado! Recompensa final: 0
Acción: 0, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: False
Acción: 1, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: True
¡Episodio terminado! Recompensa final: 0
Acción: 3, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: False
Acción: 0, Recompensa: 0, Terminado: False
Acción: 2, Recompensa: 0, Terminado: Fal

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [10]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 1


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [11]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.0%
Recompensa Promedio: 0.010
Desviación Estándar: 0.099
Total de Victorias: 1


La siguiente celda define una función para entrenar un agente.

In [12]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=500, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)

            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")



✓ Funciones de entrenamiento definidas


La siguiente celda define el agente de Q-Learning a implementar.

## Q-Learning Algorithm

1. For each state `s` and action `a`, initialize the table entry **Q(s, a)** to zero.  
2. Observe the current state **s**  
3. Repeat forever (or for each episode):
    - Select an action **a** and execute it  
    - Receive immediate reward **r**  
    - Observe the new state **s'**  
    - Update the table entry for **Q(s, a)** as follows:
      $$
      Q(s, a) \leftarrow r + \gamma \max_{a'} Q(s', a')
      $$
    - Set **s ← s'**

## Q-Learning en entornos no deterministas

En entornos estocásticos, ejecutar una acción desde un estado no garantiza llegar siempre al mismo siguiente estado, ni recibir siempre la misma recompensa. Por eso, la actualización de la tabla Q se hace de manera **promediada**, usando un factor de aprendizaje α.

La fórmula clásica de actualización es:

$$
Q(s, a) \leftarrow (1-\alpha) Q(s, a) + \alpha \big[r + \gamma \max_{a'} Q(s', a')\big]
$$

- **α (tasa de aprendizaje):** determina cuánto peso le damos a la nueva información frente a lo aprendido hasta ahora.  
- **r + γ max Q(s', a'):** estimación de la recompensa total esperada al ejecutar la acción.
- En **entornos deterministas**, se puede sobrescribir Q(s, a) directamente porque la acción siempre lleva al mismo estado y recompensa.
- En **entornos no deterministas**, se necesita promediar la experiencia, ya que la acción puede llevar a distintos estados con distintas recompensas. El valor de Q converge así al **valor esperado a largo plazo** de cada acción.


In [8]:
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self, espacio_estados, espacio_acciones):
        self.Q_table = np.zeros((espacio_estados.n, espacio_acciones.n))
        self.alpha = 0.8
        self.gamma = 0.9
        self.epsilon = 1
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.espacio_acciones = espacio_acciones
        self.tam_mapa = int(np.sqrt(espacio_estados.n))  

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy.""" #o softmax, lo cambio a softmax luego 
        if np.random.rand() < self.epsilon:
            return np.random.randint(4)  # Exploración
        else:
            return np.argmax(self.Q_table[estado])  # Explotación, accion que maximiza Q
        
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        max_q = np.max(self.Q_table[siguiente_estado])
        self.Q_table[estado, accion] = (1 - self.alpha) * self.Q_table[estado, accion] + self.alpha * (recompensa + self.gamma * max_q)
        if terminado:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def imprimirQtable(self):
        print("Tabla Q:")
        print(self.Q_table)

    def mostrar_policy(self, env):
        """
        Muestra la política aprendida en el mapa del FrozenLake.
        Flechas solo en casillas 'F' (Frozen); 'H' y 'G' se muestran tal cual.
        """
        # Mapeo de acciones a flechas: 0=izquierda, 1=abajo, 2=derecha, 3=arriba
        acciones_map = ["←", "↓", "→", "↑"]

        # Obtener el mapa original del entorno
        desc = env.unwrapped.desc.astype(str)

        # Crear copia del mapa para mostrar la política
        policy_map = desc.copy()

        # Recorrer filas y columnas del mapa
        for fila in range(self.tam_mapa):
            for col in range(self.tam_mapa):
                estado = fila * self.tam_mapa + col  # convertir fila,col a índice lineal
                if desc[fila, col] == 'F':  # solo actualizar casillas 'Frozen'
                    mejor_accion = np.argmax(self.Q_table[estado])
                    policy_map[fila, col] = acciones_map[mejor_accion]

        print("Policy aprendida:")
        for fila in policy_map:
            print(" ".join(fila))


Aqui deberán incluir código para entrenar y evaluar el agente de Q-Learning implementado.

In [9]:
# Entrenar y Evaluar el Agente Q-Learning en FrozenLake-v1 4x4 
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)

agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=100000, verbose=True)
agente.imprimirQtable()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG

Episodio 10000: Recompensa Promedio = 0.442, Longitud Promedio = 32.8
Episodio 20000: Recompensa Promedio = 0.468, Longitud Promedio = 34.5
Episodio 30000: Recompensa Promedio = 0.469, Longitud Promedio = 33.1
Episodio 40000: Recompensa Promedio = 0.456, Longitud Promedio = 33.5
Episodio 50000: Recompensa Promedio = 0.464, Longitud Promedio = 33.9
Episodio 60000: Recompensa Promedio = 0.483, Longitud Promedio = 34.3
Episodio 70000: Recompensa Promedio = 0.448, Longitud Promedio = 33.3
Episodio 80000: Recompensa Promedio = 0.452, Longitud Promedio = 34.1
Episodio 90000: Recompensa Promedio = 0.468, Longitud Promedio = 34.5
Episodio 100000: Recompensa Promedio = 0.453, Longitud Promedio = 33.8
Tabla Q:
[[1.13607312e-01 4.35162239e-03 3.70982042e-03 4.12006765e-03]
 [2.84835299e-03 1.37315349e-03 7.95155508e-06 5.00159637e-02]
 [1.97271252e-03 2.73349078e-03 3.83624536e-03 3.63289596e-02]
 [3.79767524e-03 2.98732371e-04 3.96569844e-05 7.19015517e-0

In [16]:
#env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
#estado, _ = env.reset()
#agente = AgenteQLearning(env.observation_space, env.action_space)
#print(f"FrozenLake-v1 8x8: {env.render()}")
#resultados, longitudes = entrenar_agente(agente, env, num_episodios=10000000, verbose=True)
#agente.imprimirQtable()
#resultados = evaluar_agente(agente, env, num_episodios=1000)
#imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")

La siguiente celda define el agente DQN a implementar. 

In [ ]:

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto)
        self.out = nn.Linear(tamano_oculto, tamano_salida)
    
    def forward(self, x):
        x = F.relu(self.fc1(x)) #Func. activación ReLU (Rectified Linear Unit)
        x = self.out(x)         #Capa final no usa activación
        return x                

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    
    def __init__(self,
                 espacio_estados,
                 espacio_acciones,
                 tamano_oculto=64,
                 lr=1e-3,
                 gamma=0.99,
                 epsilon=1.0,
                 epsilon_decay=0.995,
                 epsilon_min=0.01,
                 replay_size=10000,
                 batch_size=64,
                 target_update=1000,
                 train_freq=4,
                 device=None):
        """
        Args:
            espacio_estados: env.observation_space (discreto)
            espacio_acciones: env.action_space
            tamano_oculto: neuronas en la capa oculta
            lr: learning rate para el optimizador
            gamma: factor de descuento
            epsilon, epsilon_decay, epsilon_min: para epsilon-greedy
            replay_size: capacidad del buffer de replay
            batch_size: tamaño de batch para entrenar
            target_update: cada cuántos pasos copiar la target network
            train_freq: entrenar cada 'train_freq' llamadas a aprender
            device: 'cpu' o 'cuda' (None -> detecta automaticamente)
        """
        # parámetros del entorno
        self.num_states = espacio_estados.n
        self.num_actions = espacio_acciones.n
        
        # hiperparámetros
        self.tamano_oculto = tamano_oculto
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.batch_size = batch_size
        self.target_update = target_update
        self.train_freq = train_freq
        
        # device
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
        
        # redes: policy_net y target_net
        self.policy_net = DQN(self.num_states, self.tamano_oculto, self.num_actions).to(self.device)
        self.target_net = DQN(self.num_states, self.tamano_oculto, self.num_actions).to(self.device)
        
        # inicializar target igual a policy
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        # optimizador y loss
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)
        self.loss_fn = nn.MSELoss()
        
        # replay buffer
        self.replay_buffer = deque(maxlen=replay_size)
        
        # contadores
        self.step_count = 0  # para updates y frecuencia de entrenamiento
        
        # semilla opcional
        random.seed(0)
        np.random.seed(0)
        torch.manual_seed(0)
    
    def _one_hot(self, estado):
        """Convierte un entero estado -> vector one-hot tensor en device."""
        vec = np.zeros(self.num_states, dtype=np.float32)
        vec[estado] = 1.0
        return torch.from_numpy(vec).to(self.device)
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        # exploración
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        
        # explotación: predecir Q-values y tomar argmax
        self.policy_net.eval()
        with torch.no_grad():
            s_t = self._one_hot(estado).unsqueeze(0)  # shape (1, num_states)
            q_vals = self.policy_net(s_t)            # shape (1, num_actions)
            accion = torch.argmax(q_vals, dim=1).item()
        self.policy_net.train()
        return accion
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """
        Agrega la transición al buffer y entrena la red periódicamente.
        Este método es llamado por el loop de entrenamiento por paso.
        """
        # almacenar transición (s, a, r, s', done)
        self.replay_buffer.append((estado, accion, recompensa, siguiente_estado, float(terminado)))
        self.step_count += 1
        
        # Entrenar cada 'train_freq' pasos (evita entrenar en cada transición)
        if (self.step_count % self.train_freq == 0) and (len(self.replay_buffer) >= self.batch_size):
            self._train_on_batch()
        
        # actualizar target network periódicamente
        if self.step_count % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        # si terminó el episodio, actualizar epsilon (igual que en Q-table)
        if terminado:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def _train_on_batch(self):
        """Muestrea un minibatch y realiza un paso de optimización."""
        batch = random.sample(self.replay_buffer, self.batch_size)
        estados, acciones, recompensas, estados_siguientes, dones = zip(*batch)
        
        # convertir a tensores
        estados = torch.stack([self._one_hot(s) for s in estados])            # (B, num_states)
        estados_siguientes = torch.stack([self._one_hot(s) for s in estados_siguientes])  # (B, num_states)
        acciones = torch.tensor(acciones, dtype=torch.long, device=self.device)           # (B,)
        recompensas = torch.tensor(recompensas, dtype=torch.float32, device=self.device)  # (B,)
        dones = torch.tensor(dones, dtype=torch.float32, device=self.device)              # (B,)
        
        # Q(s,a) predichos por policy_net
        q_preds = self.policy_net(estados)                       # (B, num_actions)
        q_pred_values = q_preds.gather(1, acciones.unsqueeze(1)).squeeze(1)  # (B,)
        
        # target: r + gamma * max_a' Q_target(s', a')  (0 si done)
        with torch.no_grad():
            q_next = self.target_net(estados_siguientes)        # (B, num_actions)
            q_next_max, _ = q_next.max(dim=1)                  # (B,)
            q_targets = recompensas + (1.0 - dones) * (self.gamma * q_next_max)
        
        # loss y optimización
        loss = self.loss_fn(q_pred_values, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        # opcional: gradient clipping
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

    def mostrar_policy(self, env):
        acciones_map = ["←", "↓", "→", "↑"]
        desc = env.unwrapped.desc.astype(str)
        tam_mapa = int(np.sqrt(self.num_states))
        policy_map = desc.copy()
        
        self.policy_net.eval()
        with torch.no_grad():
            all_states = torch.eye(self.num_states, device=self.device)
            q_vals = self.policy_net(all_states)
            mejores = torch.argmax(q_vals, dim=1).cpu().numpy()
        self.policy_net.train()
        
        for fila in range(tam_mapa):
            for col in range(tam_mapa):
                estado = fila * tam_mapa + col
                if desc[fila, col] == 'F':
                    policy_map[fila, col] = acciones_map[mejores[estado]]
        
        print("Policy aprendida (DQN):")
        for fila in policy_map:
            print(" ".join(fila))    


Aqui deberán incluir código para entrenar y evaluar el agente de DQN 
implementado.


In [15]:
env = gym.make("FrozenLake-v1", is_slippery=True, map_name="4x4")
agente = AgenteDQN(env.observation_space, env.action_space, tamano_oculto=64)
recompensas, longitudes = entrenar_agente(agente, env, num_episodios=20000)


Episodio 2000: Recompensa Promedio = 0.479, Longitud Promedio = 34.1
Episodio 4000: Recompensa Promedio = 0.663, Longitud Promedio = 42.5
Episodio 6000: Recompensa Promedio = 0.664, Longitud Promedio = 42.7
Episodio 8000: Recompensa Promedio = 0.671, Longitud Promedio = 42.5
Episodio 10000: Recompensa Promedio = 0.678, Longitud Promedio = 43.2
Episodio 12000: Recompensa Promedio = 0.671, Longitud Promedio = 44.4
Episodio 14000: Recompensa Promedio = 0.677, Longitud Promedio = 43.7
Episodio 16000: Recompensa Promedio = 0.673, Longitud Promedio = 43.7
Episodio 18000: Recompensa Promedio = 0.658, Longitud Promedio = 43.2
Episodio 20000: Recompensa Promedio = 0.665, Longitud Promedio = 42.7


In [ ]:
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()


Agente Q-Learning - Resultados de Evaluación:
Tasa de Victorias: 72.3%
Recompensa Promedio: 0.723
Desviación Estándar: 0.448
Total de Victorias: 723
Policy aprendida (DQN):
S ↑ ↑ ↑
← H → H
↑ ↓ ← H
H → ↓ G


In [20]:
env = gym.make("FrozenLake-v1", is_slippery=True, map_name="8x8")
agente = AgenteDQN(env.observation_space, env.action_space, tamano_oculto=64)
recompensas, longitudes = entrenar_agente(agente, env, num_episodios=60000)

Episodio 6000: Recompensa Promedio = 0.195, Longitud Promedio = 73.7
Episodio 12000: Recompensa Promedio = 0.434, Longitud Promedio = 73.0
Episodio 18000: Recompensa Promedio = 0.431, Longitud Promedio = 73.6
Episodio 24000: Recompensa Promedio = 0.456, Longitud Promedio = 74.4
Episodio 30000: Recompensa Promedio = 0.422, Longitud Promedio = 75.7
Episodio 36000: Recompensa Promedio = 0.444, Longitud Promedio = 75.9
Episodio 42000: Recompensa Promedio = 0.433, Longitud Promedio = 75.5
Episodio 48000: Recompensa Promedio = 0.424, Longitud Promedio = 76.0
Episodio 54000: Recompensa Promedio = 0.430, Longitud Promedio = 75.7
Episodio 60000: Recompensa Promedio = 0.435, Longitud Promedio = 75.9


In [21]:
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()


Agente Q-Learning - Resultados de Evaluación:
Tasa de Victorias: 52.7%
Recompensa Promedio: 0.527
Desviación Estándar: 0.499
Total de Victorias: 527
Policy aprendida (DQN):
S ↑ → → → → → →
↑ ↑ ↑ ↑ ↑ → → ↓
↑ ← ↑ H ↑ ↑ → →
↑ ↑ ↑ ↑ ↑ H → →
↑ ↑ ↑ H → ↓ ↑ →
↑ H H ↑ ↑ ↑ H →
↑ H ↑ ↑ H ← H →
↑ ↑ ↑ H → → ↓ G
